In [ ]:
1

In [17]:
from nocode_robot_programming.state_decision.plots.benchmark_plot import visualize_accuracies, plot_heatmap
import numpy as np
import matplotlib.pyplot as plt
from typing import List
from pathlib import Path

def plot_heatmap(matrix: np.ndarray, x_labels: List[str], y_labels: List[str], title: str, save_path: Path, jupyter_plot: bool, line=True):
    fig = plt.figure(figsize=(max(6, len(x_labels)*0.8), max(3, len(y_labels)*0.35)))
    ax = fig.add_subplot(111)

    im = ax.imshow(matrix, aspect="auto")

    ax.set_xticks(np.arange(len(x_labels)))
    ax.set_yticks(np.arange(len(y_labels)))
    ax.set_xticklabels(x_labels, rotation=45, ha="right")
    ax.set_yticklabels(y_labels)

    ax.set_ylabel("Models")
    ax.set_title(title)

    # --- vertical separator between column groups ---
    if line:
        ax.axvline(2.5, color="black", linewidth=1.5)

    # --- group titles (positioned in axis coordinates) ---
    if line:
        ax.text(0.25, 1.00, "Tasks", transform=ax.transAxes,
                ha="center", va="bottom", fontsize=8, fontweight="bold")

        ax.text(0.75, 1.00, "Modality", transform=ax.transAxes,
                ha="center", va="bottom", fontsize=8, fontweight="bold")

    # annotate cells
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, f"{matrix[i, j]:.1f}",
                    ha="center", va="center", fontsize=10)

    fig.tight_layout()
    fig.savefig(save_path, dpi=160, bbox_inches="tight")
    plt.close(fig)

# State Estimation

In [19]:
m = np.array([
    [75.1, 71.5, 84.5, 78.9, 74.6, 74.7],
    [76.2, 72.3, 87.8, 80.8, 77.0, 75.1],
    [82.6, 74.0, 88.6, 82.8, 77.7, 80.7],
    [77.4, 76.8, 84.2, 82.1, 78.3, 76.4],
    [71.8, 71.5, 85.4, 78.5, 76.2, 71.4],
    [69.5, 67.8, 73.3, 69.2, 68.0, 72.3],
    [64.0, 59.5, 68.7, 65.3, 63.0, 61.5],
])
plot_heatmap(m, ['Peg\npick', 'Probe\nmeasure', 'Cable\nwrap', '$gst$', '$joy$', '$kin$'],
             ['dinov2 small mean', 'dinov3 small mean', 'dinov2 small concat', 'dinov2 small attn', 'dinov2 small MIL', 'SIFT', 'AEGP Multiclass'],
             "Test Accuracy (%)", "state_estimation.pdf", False)

# Anomaly

In [7]:
m = np.array([
    [65.9, 58.5, 75.7, 65.5, 70.1, 60.5],
    [61.0, 58.6, 77.9, 61.8, 71.9, 60.5],
    [78.3, 66.2, 87.9, 69.2, 76.9, 81.3],
    [79.7, 67.1, 74.8, 75.1, 79.4, 63.6],
    [24.1, 25.6, 18.7, 31.1, 16.6, 21.5],
])
plot_heatmap(m, ['Peg\npick', 'Probe\nmeasure', 'Cable\nwrap', '$gst$', '$joy$', '$kin$'],
             ['dinov2 small mean', 'dinov3 small mean', 'dinov2 small concat', 'dinov2 small attn', 'dinov2 small MIL', ], #'SIFT', 'AEGP Multiclass'], 
             "Test Accuracy (%)", "test_save.pdf", False)

# Additional

In [20]:
m = np.array([
    [92.4, 93.7, 79.8, 66.5, 55.9, 55.1, 49.0],
    [75.7, 73.4, 64.2, 58.1, 48.7, 45.3, 38.0],
    [92.4, 95.2, 77.6, 67.0, 56.2, 48.5, 43.2],
    [92.4, 94.6, 95.6, 94.0, 87.4, 82.0, 73.1],
    [92.4, 68.1, 46.2, 41.4, 28.5, 30.6, 21.4],
    [92.9, 88.7, 66.8, 62.5, 54.9, 43.7, 40.0],
    [92.4, 42.1, 27.7, 26.0, 19.0, 22.5, 15.7]
])
plot_heatmap(m, ['2 classes', '3 classes', '4 classes', '5 classes', '6 classes', '7 classes', '8 classes'],
             ['dinov2 small mean', 'dinov3 small mean', 'dinov2 small concat', 'dinov2 small attn', 'dinov2 small MIL', 'SIFT', 'AEGP Multiclass'], 
             "Test Accuracy (%)", "test_save_additional.pdf", False, line=False)

# State Estimation with 1/2 insertion

In [ ]:
from nocode_robot_programming.state_decision.plots.benchmark_plot import visualize_accuracies, plot_heatmap
import numpy as np
import matplotlib.pyplot as plt
from typing import List
from pathlib import Path

def plot_heatmap(matrix: np.ndarray, x_labels: List[str], y_labels: List[str], title: str, save_path: Path, jupyter_plot: bool):
    """Plot a heatmap. matrix can be 2-D (n_models, n_tasks) or 3-D (n_models, n_tasks, 3).
    In the 3-D case each cell has a left-to-right gradient from v[0] to max(v) and shows
    'no-inj / 1-traj / 2-traj' values as text."""
    n_rows, n_cols = matrix.shape[:2]
    fig = plt.figure(figsize=(max(6, n_cols * 0.8), max(4.5, n_rows * (0.75 if matrix.ndim == 3 else 0.6))))
    ax = fig.add_subplot(111)

    if matrix.ndim == 3:
        # Build a high-resolution gradient image: each cell gets N horizontal pixels.
        N = 60
        vmin, vmax = matrix.min(), matrix.max()
        grad = np.zeros((n_rows, n_cols * N))
        for i in range(n_rows):
            for j in range(n_cols):
                v = matrix[i, j]
                grad[i, j * N:(j + 1) * N] = np.linspace(v[0], max(v), N)
        ax.imshow(grad, aspect="auto", vmin=vmin, vmax=vmax,
                  extent=[-0.5, n_cols - 0.5, n_rows - 0.5, -0.5])
        # cell separators
        for j in range(1, n_cols):
            ax.axvline(x=j - 0.5, color="white", linewidth=0.5, alpha=0.6)
        # text annotations
        for i in range(n_rows):
            for j in range(n_cols):
                v = matrix[i, j]
                brightness = (grad[i, j * N + N // 2] - vmin) / max(vmax - vmin, 1e-9)
                color = "white" if brightness < 0.55 else "black"
                ax.text(j, i, f"{v[0]:.1f} /\n {v[1]:.1f} /\n {v[2]:.1f}",
                        ha="center", va="center", fontsize=9, color=color)
    else:
        ax.imshow(matrix, aspect="auto")
        for i in range(n_rows):
            for j in range(n_cols):
                ax.text(j, i, f"{matrix[i, j]:.1f}", ha="center", va="center", fontsize=8)

    ax.set_xticks(np.arange(n_cols))
    ax.set_yticks(np.arange(n_rows))
    ax.set_xticklabels(x_labels, rotation=45, ha="right")
    ax.set_yticklabels(y_labels)
    ax.set_xlabel("Tasks")
    ax.set_ylabel("Models")
    ax.set_title(title)
    fig.tight_layout()

    if jupyter_plot:
        ipt.save()
    else:
        fig.savefig(save_path, dpi=160, bbox_inches="tight")
        plt.close(fig)

In [ ]:
m = np.array([
    [[82.1, 91.1, 94.0], [76.0, 78.0, 85.5], [86.2, 89.4, 96.3], [82.6, 86.5, 93.1], [81.1, 84.4, 90.6], [77.7, 82.8, 88.3]],
    [[89.3, 89.0, 89.7], [73.8, 75.4, 84.7], [90.2, 94.1, 97.7], [83.6, 85.4, 90.6], [82.0, 85.1, 92.5], [81.8, 82.3, 86.6]],
    [[80.3, 87.6, 89.3], [70.7, 74.4, 83.6], [85.8, 90.0, 95.4], [79.6, 83.5, 90.7], [76.8, 82.4, 89.4], [75.9, 80.9, 85.1]],
    [[81.8, 80.5, 82.8], [71.1, 75.4, 83.0], [89.4, 96.7, 97.7], [81.8, 85.1, 89.0], [79.4, 84.1, 90.2], [75.9, 79.4, 82.8]],
    [[68.6, 86.6, 88.9], [70.2, 74.5, 75.1], [92.5, 95.0, 97.2], [79.3, 86.6, 88.6], [75.1, 86.3, 87.3], [73.8, 77.9, 79.2]],
    [[71.8, 82.1, 84.6], [66.7, 67.8, 74.7], [73.1, 77.9, 85.3], [69.7, 73.9, 81.5], [67.7, 77.4, 82.8], [71.9, 72.5, 77.0]],
    [[64.0, 64.0, 64.0], [59.5, 59.5, 59.5], [68.7, 68.7, 68.7], [65.3, 65.3, 65.3], [63.0, 63.0, 63.0], [61.5, 61.5, 61.5]]
])
plot_heatmap(m, ['Peg\npick', 'Probe\nmeasure', 'Cable\nwrap', '$gst$', '$joy$', '$kin$'],
    [
        'dinov2 small attn', 
        'dinov2 small concat', 
        'dinov2 small mean', 
        'dinov3 small mean', 
        'dinov2 small MIL', 
        'SIFT', 
        'AEGP Multiclass'
    ],
    "Test Accuracy (%)", "state_estimation.pdf", False)